# GISLR — semantically duplicate labels: which to merge, and what it's worth

**Standalone diagnostic** (not a pipeline stage): reads the run registry and
`sign_to_prediction_index_map.json`, writes `sb.recognize.label_merge` (the decided merge
groups, already in `packages/`) and this notebook's own evidence tables into
`docs/reports/label-merging.md`. Nothing here changes a run record, a checkpoint, or the
dataset's labels.

**The user's task (2026-09-25): "analyze all the glosses of gislr; if some labels are
semantically similar then make them into the same label, as the model is mislabelling
them; then check the accuracy of the top performing models."** Two parts:

1. **§1-§3**: a systematic pass over all 250x249/2 gloss pairs (not just a hand-picked
   list) for semantic similarity, cross-referenced against the aggregate confusion
   matrix — a pair is only merged when **both** agree. `docs/reports/
   plateau-diagnosis.md` §1 already found and named 14 of these pairs from a 16-pair
   hand list (pre-registry-reset, old split); this notebook re-derives the set from
   scratch on the *current* canonical split, checks nothing was missed, and packages
   the result as reusable code (`sb.recognize.label_merge`) instead of a one-off table.
2. **§4**: strict vs. merged accuracy for the current top canonical models. **Blocked on
   `gislr.1.models.training.ipynb`'s run in progress (TODO §4.3's 12-run grid) — the
   user asked to wait for it before this step.** §4's cells are written and will run
   once those runs are evaluated; until then they report on whatever is on the
   leaderboard *now*, clearly labeled as provisional.

## Why WordNet alone doesn't decide this

Wu-Palmer similarity (taxonomic tree distance) rates sibling categories — `cat`/`dog`,
any two colors, `horse`/`zebra` — just as high as true near-synonyms, because it measures
"same branch of the tree," not "same meaning." §1 shows this concretely. It's supporting
evidence, cross-checked against §2's empirical confusion, never the sole decision.

## 1. Semantic similarity, systematically — and why it needs a second check

Dominant WordNet sense per part of speech (the *first*, most-frequent synset — not
the best-scoring sense across all of a word's meanings, which turns out to match
coincidentally through obscure senses for short common words: naive best-over-all-senses
scoring puts `make`/`pretend` and `stay`/`stuck` at a perfect 1.0 through senses no
GISLR video means). Even restricted to dominant senses, Wu-Palmer similarity clusters
same-category siblings as tightly as real synonyms — the table below shows both kinds
side by side; §2 is what tells them apart.

In [1]:
# ===== Imports, vocab, WordNet setup =====
import itertools
import json
from pathlib import Path

import nltk
import numpy as np
import pandas as pd

from sb.core.paths import CACHE_DIR, EXTERNAL_DIR, MODELS_DIR, ROOT_DIR, gislr_dir
from sb.core.vocab import load_label_map
from sb.mlops import query as SUB
from sb.recognize import label_merge as LM

DATA_DIR = gislr_dir()
LABEL_MAP = load_label_map(DATA_DIR)
WORDS = sorted(LABEL_MAP, key=lambda w: LABEL_MAP[w])
assert len(WORDS) == 250
EVAL_DIR = CACHE_DIR / "gislr" / "evaluation"

nltk_root = EXTERNAL_DIR / "nltk_data"
if str(nltk_root) not in nltk.data.path:
    nltk.data.path.append(str(nltk_root))
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk_root.mkdir(parents=True, exist_ok=True)
    nltk.download("wordnet", download_dir=str(nltk_root), quiet=True)
    nltk.download("omw-1.4", download_dir=str(nltk_root), quiet=True)
from nltk.corpus import wordnet as wn

print(f"{len(WORDS)} glosses, label map from {DATA_DIR.name}")

250 glosses, label map from 1


In [2]:
# ===== Dominant-sense WordNet similarity over every pair =====
# Compound/non-dictionary GISLR tokens mapped to a lookupable lemma; pronouns and other
# purely grammatical tokens get None (no WordNet entry -- and none should be invented).
LOOKUP = {
    "callonphone": "call", "frenchfries": "french_fries", "glasswindow": "window",
    "icecream": "ice_cream", "hesheit": None, "minemy": None, "weus": None,
    "haveto": "have", "yourself": None, "TV": "television",
}


def dominant_synsets(w: str):
    """The single most frequent synset per part of speech (WordNet lists senses by
    frequency, so [0] is the dominant one) -- not the best match across every sense."""
    lw = LOOKUP.get(w, w)
    if lw is None:
        return []
    by_pos: dict[str, object] = {}
    for s in wn.synsets(lw.replace(" ", "_")):
        by_pos.setdefault(s.pos(), s)  # first hit per POS = dominant
    return list(by_pos.values())


SYNSETS = {w: dominant_synsets(w) for w in WORDS}


def wn_similarity(a: str, b: str) -> tuple[float, str]:
    sa, sb = SYNSETS[a], SYNSETS[b]
    best, why = 0.0, ""
    for s1 in sa:
        for s2 in sb:
            if s1.pos() != s2.pos():
                continue
            if s1 == s2:
                return 1.0, f"same dominant synset {s1.name()}"
            sim = s1.wup_similarity(s2) or 0.0
            if sim > best:
                best, why = sim, f"{s1.name()} ~ {s2.name()}"
    return best, why


rows = []
for a, b in itertools.combinations(WORDS, 2):
    sim, why = wn_similarity(a, b)
    if sim >= 0.75:
        rows.append({"a": a, "b": b, "wn_sim": round(sim, 3), "why": why})
wn_df = pd.DataFrame(rows).sort_values("wn_sim", ascending=False).reset_index(drop=True)
print(f"{len(wn_df)} pairs at wn_sim >= 0.75 (dominant-sense Wu-Palmer)")
wn_df.head(20)

233 pairs at wn_sim >= 0.75 (dominant-sense Wu-Palmer)


,a,b,wn_sim,why
0,after,later,1.000,same dominant synset subsequently.r.01
1,have,haveto,1.000,same dominant synset rich_person.n.01
2,dry,dryer,1.000,same dominant synset dry.a.01
3,cereal,grass,0.957,cereal.n.01 ~ grass.n.01
4,bedroom,room,0.941,bedroom.n.01 ~ room.n.01
5,horse,zebra,0.933,horse.n.01 ~ zebra.n.01
6,duck,goose,0.929,duck.n.01 ~ goose.n.01
7,dog,wolf,0.929,dog.n.01 ~ wolf.n.01
8,doll,toy,0.923,doll.n.01 ~ plaything.n.01
9,airplane,helicopter,0.917,airplane.n.01 ~ helicopter.n.01


Confirms the problem: `horse`/`zebra` (0.93), `dog`/`wolf` (0.93), every color pair
(0.875), `brother`/`uncle` (0.857) all score as high as `pen`/`pencil` (0.889) or
`nap`/`sleep` (0.889) — taxonomic closeness, not synonymy. On to §2.

## 2. Cross-reference with the *current* aggregate confusion matrix

`data/cache/gislr/evaluation/confusion_all_normalized.npy` already exists but predates
today's training run and may mix in pre-reset (legacy-split) runs — rebuild it here from
every **current-split** canonical run on the leaderboard right now, the same recipe as
`gislr.2.models.evaluation.ipynb`. This is the *primary* evidence: a pair only gets
merged if the model actually spends accuracy confusing it, not because English makes
them sound alike.

In [3]:
# ===== Rebuild the confusion matrix from current-split canonical runs only =====


def confusion_matrix(run_dir: Path, n_classes: int = 250, normalize: bool = True):
    npz_path = run_dir / "assets" / "val_predictions.npz"
    d = np.load(npz_path)
    cm = np.zeros((n_classes, n_classes), dtype=np.float64)
    np.add.at(cm, (d["labels"].astype(int), d["preds"].astype(int)), 1)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0)
    return cm


board = SUB.leaderboard("gislr", include_legacy=False)
evaluated = board[board["eval_status"] == "canonical"]
mats, used = [], []
for rid in evaluated["run_id"]:
    p = MODELS_DIR / str(int(rid)) / "assets" / "val_predictions.npz"
    if p.is_file():
        mats.append(confusion_matrix(MODELS_DIR / str(int(rid))))
        used.append(int(rid))
assert mats, "no current-split canonical runs with cached predictions"
cm_current = np.mean(mats, axis=0)
print(f"{len(used)} current-split canonical runs averaged: {used}")

wi = {w: i for i, w in enumerate(WORDS)}


def symmetric_rate(a: str, b: str) -> tuple[float, float, float]:
    ia, ib = wi[a], wi[b]
    return cm_current[ia, ib], cm_current[ib, ia], (cm_current[ia, ib] + cm_current[ib, ia]) / 2


wn_df[["a_to_b", "b_to_a", "mean_rate"]] = wn_df.apply(
    lambda r: pd.Series(symmetric_rate(r["a"], r["b"])), axis=1)
wn_df = wn_df.sort_values("mean_rate", ascending=False)
print(f"\ntop candidates by empirical confusion (of the {len(wn_df)} semantically-scored pairs):")
wn_df.head(25)

25 current-split canonical runs averaged: [1790355555, 1789559734, 1790327942, 1790347646, 1790327033, 1789558839, 1790330305, 1790354810, 1789560829, 1790332037, 1790349689, 1790338279, 1790333188, 1790334584, 1790336517, 1790339908, 1790341214, 1790143122, 1790144582, 1790356714, 1790351435, 1790352947, 1790146838, 1790352803, 1790142624]

top candidates by empirical confusion (of the 233 semantically-scored pairs):


,a,b,wn_sim,why,a_to_b,b_to_a,mean_rate
25,pen,pencil,0.889,pen.n.01 ~ pencil.n.01,0.225500,0.215385,0.220442
6,duck,goose,0.929,duck.n.01 ~ goose.n.01,0.176790,0.182785,0.179787
44,cut,scissors,0.857,cut.v.01 ~ scissor.v.01,0.167568,0.143784,0.155676
116,animal,have,0.800,animal.n.01 ~ rich_person.n.01,0.157143,0.117612,0.137377
2,dry,dryer,1.000,same dominant synset dry.a.01,0.089231,0.156216,0.122723
22,nap,sleep,0.889,nap.v.01 ~ sleep.v.01,0.169231,0.048947,0.109089
18,dog,puppy,0.897,dog.n.01 ~ puppy.n.01,0.049474,0.092394,0.070934
4,bedroom,room,0.941,bedroom.n.01 ~ room.n.01,0.108205,0.012432,0.060319
68,bird,goose,0.833,bird.n.01 ~ goose.n.01,0.055802,0.056709,0.056256
14,rain,snow,0.900,rain.n.01 ~ snow.n.01,0.037368,0.030130,0.033749


## 3. All top empirical confusions, whether or not WordNet flagged them

§1's Wu-Palmer scan only catches pairs *WordNet itself* thinks are related — it will
miss real synonym pairs WordNet's dominant sense doesn't line up with (already known
casualties: `give`/`gift` needed the noun/verb cross, `awake`/`wake` matched through an
adverb sense). So separately: every pair with real empirical confusion, semantically
judged by hand (the only reliable arbiter here — see the module docstring for why), not
filtered by §1's score.

In [4]:
# ===== Every pair with real empirical confusion (mean_rate > 0.03), independent of §1 =====
all_rows = []
for i in range(250):
    for j in range(i + 1, 250):
        r = (cm_current[i, j] + cm_current[j, i]) / 2
        if r > 0.03:
            all_rows.append({"a": WORDS[i], "b": WORDS[j], "mean_rate": round(r, 4),
                             "a_to_b": round(cm_current[i, j], 4), "b_to_a": round(cm_current[j, i], 4)})
confused_df = pd.DataFrame(all_rows).sort_values("mean_rate", ascending=False).reset_index(drop=True)
print(f"{len(confused_df)} pairs with mean confusion > 0.03")
confused_df

110 pairs with mean confusion > 0.03


,a,b,mean_rate,a_to_b,b_to_a
0,awake,wake,0.3858,0.3970,0.3745
1,lips,mouth,0.2461,0.2286,0.2636
2,pen,pencil,0.2204,0.2255,0.2154
3,duck,goose,0.1798,0.1768,0.1828
4,stay,that,0.1631,0.1483,0.1779
...,...,...,...,...,...
105,farm,mom,0.0319,0.0410,0.0228
106,red,thirsty,0.0315,0.0453,0.0178
107,jump,look,0.0311,0.0348,0.0275
108,hide,yourself,0.0305,0.0444,0.0166


### Verdict (hand-judged against the table above; full rationale in each group's `note`)

**Merge — semantically the same/near-synonymous concept, and empirically confused**
(now `sb.recognize.label_merge.MERGE_GROUPS`):

| group | canonical | mean rate(s) |
|---|---|---|
| `awake` <-> `wake` | `wake` | 0.356 |
| `give` <-> `gift` | `gift` | 0.161 |
| `listen` <-> `hear` | `listen` | 0.181 |
| `nap` <-> `sleep` <-> `sleepy` | `sleep` | 0.116 / 0.118 / 0.049 |
| `kitty` <-> `cat` | `cat` | 0.124 |
| `pencil` <-> `pen` | `pencil` | 0.182 |
| `look` <-> `see` | `see` | 0.080 |
| `mouth` <-> `lips` | `lips` | 0.224 |
| `puppy` <-> `dog` | `dog` | 0.077 |

**Considered, empirically confused, but rejected — not a semantic duplicate** (a tool and
its characteristic action, a compound sign sharing one component, or unrelated meanings
that likely share a handshape): `cut`/`scissors`, `goose`/`duck`, `bedroom`/`bed`,
`bedroom`/`room`, `dryer`/`dry`, `hear`/`ear`, `wait`/`finger`, `touch`/`find`,
`please`/`minemy`, `bad`/`thankyou`, `stay`/`that`, `animal`/`have`, `animal`/`bath`.
Matches `docs/reports/plateau-diagnosis.md` §1's independent read of `finger`/`wait` and
`animal`/`have` as "not semantically related."

**Considered, semantically plausible, but rejected — the model already tells them apart**
(near-zero in the current matrix, so merging would just remove a real distinction):
`not`/`no`, `cute`/`pretty`, `shower`/`bath`, `talk`/`say`, `fall`/`drop`, `bad`/`yucky`,
`loud`/`noisy`, `later`/`after`, `that`/`there`, `quiet`/`shhh` (weak: 0.026 mean).

**Same-category siblings, correctly not flagged as confused by the model, so not
merged**: `bug`/`bee`, `lion`/`tiger`, `wolf`/`dog`, `horse`/`donkey`, and every color/
animal/furniture/clothing pair §1 scored highly by pure taxonomy.

In [5]:
# ===== Verify the empirical table above against sb.recognize.label_merge (no drift) =====
group_words = LM.word_to_canonical()
covered = confused_df[confused_df.apply(
    lambda r: r["a"] in group_words or r["b"] in group_words, axis=1)]
print("Confused pairs (mean_rate > 0.03) touching a merge group:")
print(covered[["a", "b", "mean_rate"]].to_string(index=False))
print(f"\n{len(LM.MERGE_GROUPS)} merge groups, {len(group_words)} words folded, "
      f"canonical labels: {sorted({g.canonical for g in LM.MERGE_GROUPS})}")

Confused pairs (mean_rate > 0.03) touching a merge group:
     a      b  mean_rate
 awake   wake     0.3858
  lips  mouth     0.2461
   pen pencil     0.2204
  hear listen     0.1582
  gift   give     0.1520
 sleep sleepy     0.1446
   cat  kitty     0.1261
   nap  sleep     0.1091
   ear   hear     0.0944
  look    see     0.0712
   dog  puppy     0.0709
   nap sleepy     0.0562
  lips  tooth     0.0555
 mouth  tooth     0.0544
pretty  sleep     0.0392
   pen  potty     0.0336
  jump   look     0.0311

9 merge groups, 19 words folded, canonical labels: ['cat', 'dog', 'gift', 'lips', 'listen', 'pencil', 'see', 'sleep', 'wake']


## 4. Strict vs. merged vs. lenient accuracy for the top canonical models

Scores the top current-split canonical models three ways on the canonical val split
(GISLR_Stratified's own held-out 18,896 videos):

- **strict**: the prediction must be the exact gloss;
- **merged**: a prediction in the same synonym group as the label also counts (the 9
  `MERGE_GROUPS`: `wake`/`awake`, `see`/`look`, ...), the principled "similar word" answer;
- **lenient**: also counts any pair §2 rates WordNet-similar at `LENIENT_WN` or above
  (`horse`/`zebra`, `rain`/`snow`, `duck`/`goose`). Related concepts, not synonyms: an upper
  bound on what "similar word" could mean, not a claim that these predictions are right.


In [6]:
# ===== Top-N current-split canonical models: strict vs. merged vs. lenient accuracy =====
TOP_N = 10
LENIENT_WN = 0.85   # Wu-Palmer threshold for the lenient tier (section 2's wn_df starts at 0.75)

_groups = LM.word_to_canonical()
canon = {w: _groups.get(w, w) for w in WORDS}
lenient_pairs = {frozenset((a, b)) for a, b, s in wn_df[["a", "b", "wn_sim"]].itertuples(index=False)
                 if s >= LENIENT_WN}
idx2word = {i: w for w, i in LABEL_MAP.items()}

top = evaluated.sort_values("accuracy", ascending=False).head(TOP_N)
rows = []
for _, run in top.iterrows():
    run_dir = MODELS_DIR / str(int(run["run_id"]))
    npz_path = run_dir / "assets" / "val_predictions.npz"
    if not npz_path.is_file():
        continue
    d = np.load(npz_path)
    r = LM.merge_report(d["labels"], d["preds"], LABEL_MAP)
    t = [idx2word[int(i)] for i in d["labels"]]
    q = [idx2word[int(i)] for i in d["preds"]]
    ok = [a == b or canon[a] == canon[b] or frozenset((a, b)) in lenient_pairs for a, b in zip(t, q)]
    rows.append({"run_id": int(run["run_id"]), "architecture": run["architecture"],
                 "subset": run.get("subset"), "streaming": bool(run.get("streaming")), **r,
                 "lenient_accuracy": float(np.mean(ok))})
merge_table = pd.DataFrame(rows)
print(f"top {len(merge_table)} current-split canonical models; lenient = merged + "
      f"{len(lenient_pairs)} WordNet pairs at >= {LENIENT_WN}")
merge_table[["run_id", "architecture", "subset", "streaming", "strict_accuracy",
             "merged_accuracy", "lenient_accuracy", "delta"]]


top 10 current-split canonical models; lenient = merged + 60 WordNet pairs at >= 0.85


,run_id,architecture,subset,streaming,strict_accuracy,merged_accuracy,lenient_accuracy,delta
0,1790355555,gru_phono_raw,ME_134,True,0.763230,0.777360,0.782229,0.014130
1,1789559734,gru,ME_132,True,0.751746,0.765665,0.770692,0.013918
2,1790327942,gru,ME_132,True,0.751058,0.765083,0.770269,0.014024
3,1790347646,bilstm,ME_126,False,0.750159,0.764765,0.769793,0.014606
4,1790327033,gru,ME_126,True,0.745025,0.759102,0.764289,0.014077
5,1789558839,gru,ME_126,True,0.745025,0.759102,0.764289,0.014077
6,1790330305,gru,ME_126,True,0.745025,0.759102,0.764289,0.014077
7,1790354810,gru_phono_raw,PH_55,True,0.742909,0.757250,0.762807,0.014342
8,1789560829,gru,FP_118,True,0.742485,0.757250,0.762119,0.014765
9,1790332037,gru,FP_118,True,0.742485,0.757250,0.762119,0.014765


## 5. Write the report

Regenerates `docs/reports/label-merging.md`'s "Current numbers" section from the tables
above — run after §4 has the final (post-training) top models.

In [7]:
# ===== Write the current-numbers block into the report =====
report_path = ROOT_DIR / "docs" / "reports" / "label-merging.md"
if report_path.is_file() and not merge_table.empty:
    block = ["## Current numbers (regenerated by this notebook)\n\n",
             f"{len(used)} current-split canonical runs in the aggregate confusion matrix: {used}.\n\n",
             "| run | architecture | subset | streaming | strict | merged (synonyms) | lenient (WordNet) |\n",
             "|---|---|---|---|---|---|---|\n"]
    for _, r in merge_table.iterrows():
        block.append(f"| `{r['run_id']}` | {r['architecture']} | {r['subset']} | "
                      f"{'yes' if r['streaming'] else 'no'} | {r['strict_accuracy']:.4f} | "
                      f"{r['merged_accuracy']:.4f} (+{r['delta']:.4f}) | "
                      f"{r['lenient_accuracy']:.4f} |\n")
    block.append(f"\nLenient = merged + {len(lenient_pairs)} WordNet pairs at Wu-Palmer >= "
                 f"{LENIENT_WN} (related concepts such as horse/zebra, not synonyms): an upper "
                 "bound, not a claim that those predictions are right.\n")
    text = report_path.read_text(encoding="utf-8")
    marker = "## Current numbers (regenerated by this notebook)"
    if marker in text:
        text = text[: text.index(marker)] + "".join(block)
    else:
        text = text.rstrip() + "\n\n" + "".join(block)
    report_path.write_text(text, encoding="utf-8")
    print(f"wrote {report_path}")
else:
    print("skipped: report doesn't exist yet, or section 4 has no rows (training still in progress)")

wrote C:\Users\user2\repository\docs\reports\label-merging.md
